# 01 · Token embeddings

Turn a sentence into token IDs and use them to look up vectors in an embedding
table. Explore the array shapes and why repeated tokens receive the same vector.


In [1]:
from pathlib import Path

import numpy as np
from model_tokenizer import Tokenizer

# Load the vocabulary that will determine the number of embedding rows.
tokenizer_path = Path("../../../trained-models/tokenizers/simple-english.json").resolve()
tokenizer = Tokenizer.load(tokenizer_path)


## 1. Turn a sentence into row numbers

We reuse the trained tokenizer rather than inventing a second vocabulary.
Each element in `token_ids` will select one row of our table. Spaces and
punctuation are tokens too; a token is not necessarily a whole word.


In [2]:
text = "Once upon a time, there was a fluffy bunny."
token_ids = np.array(tokenizer.encode(text), dtype=np.int64)

print(text)
print("Token IDs:", token_ids)
print("Token ID shape:", token_ids.shape)
for position, token_id in enumerate(token_ids):
    piece = tokenizer.vocabulary[int(token_id)]
    print(f"{position:2}: ID {token_id:4} -> {piece!r}")


Once upon a time, there was a fluffy bunny.
Token IDs: [ 400   32  405   32   97   32  365   44   32  372   32  270   32   97
   32  102  940   32 1039   46]
Token ID shape: (20,)
 0: ID  400 -> b'Once'
 1: ID   32 -> b' '
 2: ID  405 -> b'upon'
 3: ID   32 -> b' '
 4: ID   97 -> b'a'
 5: ID   32 -> b' '
 6: ID  365 -> b'time'
 7: ID   44 -> b','
 8: ID   32 -> b' '
 9: ID  372 -> b'there'
10: ID   32 -> b' '
11: ID  270 -> b'was'
12: ID   32 -> b' '
13: ID   97 -> b'a'
14: ID   32 -> b' '
15: ID  102 -> b'f'
16: ID  940 -> b'luffy'
17: ID   32 -> b' '
18: ID 1039 -> b'bunny'
19: ID   46 -> b'.'


## 2. Create one vector for every vocabulary entry

Let $V$ be the vocabulary size and $D$ the embedding width. Our embedding table
$E$ has shape $(V, D)$: one row per token, one column per vector coordinate.
Here that is **(2048, 4)**, including a row for the document-end control token.

`np.random.default_rng(42)` creates a random-number generator with a fixed seed,
so rerunning gives the same example. `normal` draws numbers around zero; the
standard deviation 0.02 keeps these initial values small. This scale is an
illustration, not a settled model initialization rule. `astype(np.float32)`
stores each number as a 32-bit floating-point value.

These are **untrained parameters**, not meaningful descriptions of tokens.
The four coordinates have no assigned labels such as “animal” or “size”.
Training will eventually adjust the numbers to help predict the next token.


In [3]:
embedding_dim = 4
rng = np.random.default_rng(42)

# Initialize the table once, not once per token or per lookup.
embedding_table = rng.normal(
    loc=0.0, scale=0.02, size=(tokenizer.vocab_size, embedding_dim)
).astype(np.float32)

np.set_printoptions(precision=4, suppress=True)
print("Embedding table shape:", embedding_table.shape)
print("Data type:", embedding_table.dtype)
print("First token ID:", token_ids[0])
print("Its table row:", embedding_table[token_ids[0]])


Embedding table shape: (2048, 4)
Data type: float32
First token ID: 400
Its table row: [-0.0011  0.0227  0.0253 -0.0284]


## 3. Look up the sentence's embeddings

The whole forward calculation is:

`token_embeddings = embedding_table[token_ids]`

NumPy's integer-array indexing selects one row for each ID, in sentence order.
For token position $t$ and coordinate $j$:

$$X_{t,j} = E_{\mathrm{token\_ids}[t],j}$$

If the sentence has $T$ tokens, the result has shape $(T, D)$. This example
has no batch dimension: rows are positions in one sentence, columns are the
four embedding coordinates. A lookup does not multiply by the token ID.


In [4]:
# Select the table rows in the same order as the input tokens.
token_embeddings = embedding_table[token_ids]

print("Input shape:", token_ids.shape)
print("Output shape:", token_embeddings.shape)
for token_id, vector in zip(token_ids, token_embeddings):
    piece = tokenizer.vocabulary[int(token_id)]
    print(f"{piece!r:12} ID {token_id:4} -> {vector}")


Input shape: (20,)
Output shape: (20, 4)
b'Once'      ID  400 -> [-0.0011  0.0227  0.0253 -0.0284]
b' '         ID   32 -> [-0.0128 -0.0185 -0.0078 -0.0275]
b'upon'      ID  405 -> [0.0115 0.0009 0.0152 0.0235]
b' '         ID   32 -> [-0.0128 -0.0185 -0.0078 -0.0275]
b'a'         ID   97 -> [ 0.0055 -0.0283 -0.0462  0.0011]
b' '         ID   32 -> [-0.0128 -0.0185 -0.0078 -0.0275]
b'time'      ID  365 -> [ 0.0193 -0.0145 -0.0171  0.0092]
b','         ID   44 -> [ 0.0316  0.0032 -0.0024  0.0057]
b' '         ID   32 -> [-0.0128 -0.0185 -0.0078 -0.0275]
b'there'     ID  372 -> [ 0.0066  0.0036 -0.0123 -0.0005]
b' '         ID   32 -> [-0.0128 -0.0185 -0.0078 -0.0275]
b'was'       ID  270 -> [ 0.0049 -0.003   0.0087  0.0012]
b' '         ID   32 -> [-0.0128 -0.0185 -0.0078 -0.0275]
b'a'         ID   97 -> [ 0.0055 -0.0283 -0.0462  0.0011]
b' '         ID   32 -> [-0.0128 -0.0185 -0.0078 -0.0275]
b'f'         ID  102 -> [-0.0399 -0.0259 -0.0296 -0.0467]
b'luffy'     ID  940 -> [-0.0089  0

## 4. The same token selects the same row

Our sentence repeats the space token. Every occurrence gets exactly the same
vector because we are looking up the same table row. The lookup alone has no
notion of position or surrounding words; later components will supply that.

Below we compare the first two occurrences of the space token. This tokenizer
represents a single space as one token.


In [5]:
space_id = tokenizer.encode(" ")[0]
space_positions = np.flatnonzero(token_ids == space_id)
first_position, second_position = space_positions[:2]

print("Space token ID:", space_id)
print("Positions:", space_positions)
print("First space vector:", token_embeddings[first_position])
print("Second space vector:", token_embeddings[second_position])
print("Exactly equal:", np.array_equal(
    token_embeddings[first_position], token_embeddings[second_position]
))


Space token ID: 32
Positions: [ 1  3  5  8 10 12 14 17]
First space vector: [-0.0128 -0.0185 -0.0078 -0.0275]
Second space vector: [-0.0128 -0.0185 -0.0078 -0.0275]
Exactly equal: True


## Pause here

Try changing `text` and running the notebook again. The sentence length and
selected rows change, but the seeded embedding table stays the same.

Before moving on, trace one ID from the tokenizer output to its table row and
then to its row in `token_embeddings`. Can you explain why repeated spaces
have identical vectors, even at different positions?

We have built the token lookup, not trained useful representations or built a
language model. Position information is the next separate concept to discuss.
